# Semantic Tool Discovery with the AgentCore Tool Search Plugin for Strands Agents

## 1. Introduction

This notebook demonstrates how to build a **Strands agent** that interacts with **Amazon Bedrock AgentCore Gateway** to access travel-domain tools via semantic tool search. You will deploy a plain Lambda function containing travel tools, register it with AgentCore Gateway as a tool target, and create a Strands agent that dynamically discovers and invokes the right tools based on user intent.

### What you will learn

- **Deploy a Lambda function** as a tool target for AgentCore Gateway
- **Register the function** with AgentCore Gateway so tools become discoverable
- **Create a Strands agent** with `AgentCoreToolSearchPlugin` for semantic tool discovery
- **Invoke the agent** across multiple travel domains (flights, hotels, car rentals, restaurants, currency, loyalty programs)

### Architecture Overview

The architecture follows this flow:

<pre>
┌─────────────────┐    hook       ┌───────────────────────────┐   MCP/search   ┌─────────────────────┐            ┌──────────────────┐
│  Strands Agent  │──────────────▶│  AgentCoreToolSearchPlugin│───────────────▶│  AgentCore Gateway  │            │  Lambda Function │
│  (LLM reasoning)│◀── tools ─────│ (semantic tool discovery) │◀── tools ──────│  (MCP protocol)     │            |  (travel tools)  │
│                 │               └───────────────────────────┘                │                     │            │                  │
│                 │─────────────── MCP/tool call ─────────────────────────────▶│                     │───invoke──▶│                  │
│                 │◀────────────── tool result ─────────────────────────────── │                     │◀─ result ──│                  │
└─────────────────┘                                                            └─────────────────────┘            └──────────────────┘
</pre>


## 2. Prerequisites

Before running this notebook, ensure you have the following:

- **AWS Account** with appropriate permissions configured
- **Python 3.12+** installed
- **AWS CLI** installed and configured with valid credentials
- **boto3** Python SDK installed
- **IAM permissions** for the calling identity:
  - `lambda:*` - Create, invoke, and delete Lambda functions
  - `iam:CreateRole`, `iam:AttachRolePolicy`, `iam:DetachRolePolicy`, `iam:DeleteRole` - Manage Lambda execution role
  - `bedrock:*` - Access Amazon Bedrock models for agent reasoning
  - `bedrock-agentcore:*` - Register tools and connect to AgentCore Gateway

Running this notebook will create AWS resources that may incur charges:

- **AWS Lambda** - Function invocations during deployment verification and agent tool calls
- **Amazon Bedrock** - Model inference for agent reasoning and intent classification
- **AgentCore Gateway** - Gateway usage for tool registration and MCP-based tool invocations

To avoid ongoing costs, be sure to run the **Cleanup** section at the end of this notebook to delete all created resources.

## 3. Environment Setup

In this section we install the required Python packages, verify that your AWS credentials are configured correctly, and define all configurable parameters in one place.

### Install Required Packages

The following cell installs the Python packages needed for this tutorial:

- `strands-agents` - The Strands SDK for building AI agents
- `strands-agents-tools` - Pre-built tools for Strands agents
- `boto3` - AWS SDK for Python
- `bedrock-agentcore[strands-agents]` - AgentCore SDK with Strands plugin for semantic tool discovery
- `mcp-proxy-for-aws` - IAM-authenticated MCP transport for connecting to AgentCore Gateway

In [ ]:
%pip install --quiet strands-agents strands-agents-tools boto3 "bedrock-agentcore[strands-agents]" mcp-proxy-for-aws

### Verify AWS Credentials

Before proceeding, let's verify that your AWS credentials are configured correctly and confirm which account and region you are operating in.

In [ ]:
import boto3

sts = boto3.client("sts")
identity = sts.get_caller_identity()
print(f"Account: {identity['Account']}")
print(f"ARN: {identity['Arn']}")
print("[OK] AWS credentials verified successfully")

### Configuration

All configurable parameters for this tutorial are defined in the cell below. This is the single place where you can customize the behavior for your environment.

The AgentCore Gateway will be created later in the notebook - no pre-existing gateway is required.

In [ ]:
import os

# Configuration - modify these values for your environment
AWS_REGION = os.environ.get("AWS_REGION", "us-east-1")
LAMBDA_FUNCTION_NAME = "agentcore-travel-tools"
LAMBDA_ROLE_NAME = "agentcore-travel-tools-role"
MODEL_ID = "us.anthropic.claude-sonnet-4-20250514-v1:0"

print(f"Region: {AWS_REGION}")
print(f"Lambda Function: {LAMBDA_FUNCTION_NAME}")
print(f"Model: {MODEL_ID}")

## 4. Deploy the Lambda Function

In this section we deploy a **plain Lambda function** as a tool target for AgentCore Gateway. The function contains 34 travel-domain tools across 9 domains (flights, hotels, car rentals, restaurants, currency, loyalty, weather, activities, and trip planning).

**Important:** This Lambda is NOT an MCP server. It is a standard Lambda function that receives tool arguments as input and returns results. AgentCore Gateway handles the MCP protocol translation - it wraps the Lambda as a tool target and routes MCP tool calls from the agent to the function automatically.

### Create IAM Execution Role

The Lambda function needs an IAM execution role to run. This role grants the Lambda service permission to assume it, and we attach the `AWSLambdaBasicExecutionRole` managed policy for CloudWatch Logs access.

In [ ]:
import json
import time

iam = boto3.client("iam", region_name=AWS_REGION)

trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "lambda.amazonaws.com"},
            "Action": "sts:AssumeRole"
        }
    ]
}

try:
    role_response = iam.create_role(
        RoleName=LAMBDA_ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description="Execution role for AgentCore travel tools Lambda"
    )
    role_arn = role_response["Role"]["Arn"]
    print(f"\u2705 Created IAM role: {LAMBDA_ROLE_NAME}")
    print(f"   ARN: {role_arn}")
    
    # Attach basic execution policy for CloudWatch Logs
    iam.attach_role_policy(
        RoleName=LAMBDA_ROLE_NAME,
        PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole"
    )
    print("\u2705 Attached AWSLambdaBasicExecutionRole policy")
    
    # Wait for IAM role propagation
    print("\n\u23f3 Waiting 10 seconds for IAM role propagation...")
    time.sleep(10)
    print("\u2705 Role propagation complete")
    
except iam.exceptions.EntityAlreadyExistsException:
    role_arn = f"arn:aws:iam::{identity['Account']}:role/{LAMBDA_ROLE_NAME}"
    print(f"\u2139\ufe0f  IAM role already exists: {LAMBDA_ROLE_NAME}")
    print(f"   ARN: {role_arn}")

### Deploy the Lambda Function

We package `lambda/travel_tools.py` into a zip archive and deploy it. If the function already exists (from a previous run), we update its code instead.


In [ ]:
import zipfile
import io

lambda_client = boto3.client("lambda", region_name=AWS_REGION)

# Package the Lambda function
zip_buffer = io.BytesIO()
with zipfile.ZipFile(zip_buffer, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write("lambda/travel_tools.py", "travel_tools.py")
zip_buffer.seek(0)

try:
    response = lambda_client.create_function(
        FunctionName=LAMBDA_FUNCTION_NAME,
        Runtime="python3.12",
        Role=role_arn,
        Handler="travel_tools.lambda_handler",
        Code={"ZipFile": zip_buffer.read()},
        Description="Travel domain tools for AgentCore Gateway",
        Timeout=30,
        MemorySize=256
    )
    print(f"\u2705 Lambda function created: {LAMBDA_FUNCTION_NAME}")
    print(f"   ARN: {response['FunctionArn']}")
    lambda_arn = response["FunctionArn"]
except lambda_client.exceptions.ResourceConflictException:
    print(f"\u2139\ufe0f  Lambda function already exists: {LAMBDA_FUNCTION_NAME}")
    # Update the function code instead
    zip_buffer.seek(0)
    lambda_client.update_function_code(
        FunctionName=LAMBDA_FUNCTION_NAME,
        ZipFile=zip_buffer.read()
    )
    func_info = lambda_client.get_function(FunctionName=LAMBDA_FUNCTION_NAME)
    lambda_arn = func_info["Configuration"]["FunctionArn"]
    print(f"   Updated function code. ARN: {lambda_arn}")

### Verify Lambda Deployment

Let's verify the deployment by invoking the Lambda function with a test event. We'll call the `get_supported_currencies` tool to confirm the function is responding correctly.

In [ ]:
# Verify the Lambda function by invoking a test tool call
# For direct invocation, we pass tool_name in the event (fallback mode)
# In production, the gateway passes tool name via context.client_context.custom
test_event = {
    "tool_name": "get_supported_currencies"
}

response = lambda_client.invoke(
    FunctionName=LAMBDA_FUNCTION_NAME,
    Payload=json.dumps(test_event)
)

payload = json.loads(response["Payload"].read())
print("[OK] Lambda function verified successfully!")
print(f"   Supported currencies: {payload['total']} currencies available")
print(f"   Sample: {[c['code'] for c in payload['currencies'][:5]]}")


## 5. Create AgentCore Gateway and Register Tools

**Amazon Bedrock AgentCore Gateway** is a managed service that provides an MCP-compatible endpoint for your agents. In this section, we'll:

1. **Create a new gateway** - This gives us a managed MCP endpoint
2. **Register the Lambda function** as a tool target with the gateway
3. **Verify** the tools are discoverable


### Create the AgentCore Gateway

First, we create a new AgentCore Gateway. This gives us a managed MCP endpoint that agents can connect to. The gateway will handle MCP protocol translation for any tool targets we register with it.

In [ ]:
# Create an AgentCore Gateway
agentcore_control_client = boto3.client("bedrock-agentcore-control", region_name=AWS_REGION)

GATEWAY_NAME = "agentcore-travel-gateway"
GATEWAY_ROLE_NAME = "agentcore-travel-gateway-role"

# Create IAM role for the gateway
gateway_trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
            "Action": "sts:AssumeRole"
        }
    ]
}

try:
    gw_role_response = iam.create_role(
        RoleName=GATEWAY_ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(gateway_trust_policy),
        Description="Role for AgentCore Gateway to invoke Lambda targets"
    )
    gateway_role_arn = gw_role_response["Role"]["Arn"]
    # Allow the gateway to invoke Lambda functions
    iam.attach_role_policy(
        RoleName=GATEWAY_ROLE_NAME,
        PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaRole"
    )
    print(f"[OK] Created gateway role: {GATEWAY_ROLE_NAME}")
    time.sleep(10)
except iam.exceptions.EntityAlreadyExistsException:
    gateway_role_arn = f"arn:aws:iam::{identity['Account']}:role/{GATEWAY_ROLE_NAME}"
    print(f"[INFO] Gateway role already exists: {GATEWAY_ROLE_NAME}")

# Create the gateway
try:
    gateway_response = agentcore_control_client.create_gateway(
        name=GATEWAY_NAME,
        roleArn=gateway_role_arn,
        authorizerType="AWS_IAM",
        protocolType="MCP",
        protocolConfiguration={
            "mcp": {
                "searchType": "SEMANTIC"
            }
        },
        description="Travel domain gateway for semantic tool search demo"
    )
    gateway_id = gateway_response["gatewayId"]
    GATEWAY_ENDPOINT = gateway_response["gatewayUrl"]
    print(f"[OK] AgentCore Gateway created: {GATEWAY_NAME}")
    print(f"   Gateway ID: {gateway_id}")
    print(f"   Endpoint: {GATEWAY_ENDPOINT}")
except Exception as e:
    if "already exists" in str(e).lower() or "conflict" in str(e).lower():
        gateways = agentcore_control_client.list_gateways()
        for gw in gateways.get("items", []):
            if gw.get("name") == GATEWAY_NAME:
                gateway_id = gw["gatewayId"]
                gw_detail = agentcore_control_client.get_gateway(gatewayIdentifier=gateway_id)
                GATEWAY_ENDPOINT = gw_detail["gatewayUrl"]
                break
        print(f"[INFO] Gateway already exists: {GATEWAY_NAME}")
        print(f"   Gateway ID: {gateway_id}")
        print(f"   Endpoint: {GATEWAY_ENDPOINT}")
    else:
        print(f"[ERROR] Failed to create gateway: {e}")
        print("\nTroubleshooting:")
        print("  1. Ensure your IAM identity has bedrock-agentcore-control permissions")
        print("  2. Check that AgentCore Gateway is available in your region")
        raise


### Register Lambda as Tool Target

Now we register our Lambda function with the gateway as a tool target. This makes the Lambda's tools discoverable by any agent that connects to this gateway via MCP.

In [ ]:
# Load tool schemas from file
with open("lambda/tool_schemas.json", "r") as f:
    tool_schemas = json.load(f)

print(f"Loaded {len(tool_schemas)} tool schemas from lambda/tool_schemas.json")

try:
    register_response = agentcore_control_client.create_gateway_target(
        gatewayIdentifier=gateway_id,
        name=LAMBDA_FUNCTION_NAME,
        targetConfiguration={
            "mcp": {
                "lambda": {
                    "lambdaArn": lambda_arn,
                    "toolSchema": {
                        "inlinePayload": tool_schemas
                    }
                }
            }
        },
        credentialProviderConfigurations=[
            {
                "credentialProviderType": "GATEWAY_IAM_ROLE"
            }
        ],
        description="Travel domain tools - flights, hotels, car rentals, restaurants, currency, loyalty, weather, activities, trip planning"
    )
    target_id = register_response.get("targetId", "registered")
    print(f"[OK] Lambda registered with AgentCore Gateway")
    print(f"   Target ID: {target_id}")
    print(f"   Tools registered: {len(tool_schemas)}")
except Exception as e:
    if "already exists" in str(e).lower() or "conflict" in str(e).lower():
        print(f"[INFO] Tool target already registered: {LAMBDA_FUNCTION_NAME}")
        print(f"   Continuing with existing registration...")
    else:
        print(f"[ERROR] Registration failed: {e}")
        print("\nTroubleshooting:")
        print("  1. Verify the gateway was created successfully")
        print("  2. Ensure your IAM identity has bedrock-agentcore-control permissions")
        print("  3. Verify the Lambda ARN is correct")
        raise


### Verify Registration

Let's verify the registration succeeded by listing all tool targets registered with the gateway. Our Lambda function should appear in the list.

In [ ]:
# Wait for registration to propagate, then verify
print("Waiting for target registration to propagate...")
time.sleep(5)

try:
    tools_response = agentcore_control_client.list_gateway_targets(
        gatewayIdentifier=gateway_id
    )
    targets = tools_response.get("items", [])
    print(f"[OK] Gateway has {len(targets)} registered target(s):")
    for target in targets:
        print(f"   - {target.get('name', 'unknown')}: {target.get('description', 'No description')[:60]}")
except Exception as e:
    print(f"[ERROR] Failed to list gateway targets: {e}")


## 6. Create the Strands Agent

Now that our tools are registered with AgentCore Gateway, we'll create a Strands Agent that can discover and invoke them. This involves two components:

1. **MCPClient** - Connects to AgentCore Gateway using IAM-authenticated Streamable HTTP transport
2. **AgentCoreToolSearchPlugin** - Uses the gateway's built-in `x_amz_bedrock_agentcore_search` tool to semantically discover relevant tools before each model invocation

The plugin hooks into the agent lifecycle: on each invocation, it derives user intent from conversation history, searches the gateway for matching tools, and loads only those tools for the model call. Previously loaded tools are cleared before each search, so the agent always has the most relevant set.

### MCPClient

The `MCPClient` connects to AgentCore Gateway using **IAM-authenticated Streamable HTTP** transport. This is how the agent communicates with the gateway to discover and invoke tools. The connection uses AWS Signature V4 for authentication against the `bedrock-agentcore` service.

In [ ]:
from strands.tools.mcp import MCPClient
from mcp_proxy_for_aws.client import aws_iam_streamablehttp_client

# Create MCP client connected to AgentCore Gateway
mcp_client = MCPClient(
    lambda: aws_iam_streamablehttp_client(
        endpoint=GATEWAY_ENDPOINT,
        aws_region=AWS_REGION,
        aws_service="bedrock-agentcore"
    )
)

mcp_client.start()
print("[OK] MCPClient connected to AgentCore Gateway")
print(f"   Endpoint: {GATEWAY_ENDPOINT}")
print(f"   Region: {AWS_REGION}")

### AgentCoreToolSearchPlugin

The `AgentCoreToolSearchPlugin` is powered by AgentCore Gateway's built-in **semantic search** capability (`x_amz_bedrock_agentcore_search` tool). On each agent invocation, the plugin:

1. **Derives intent** - An `IntentProvider` sends the last N messages from conversation history to an LLM to produce a concise intent string
2. **Searches the gateway** - The intent is passed to the gateway's search tool to find matching tools from all registered targets
3. **Loads tools** - Only the matched tools are loaded for that model call (previously loaded tools are cleared)

By default, the intent classifier reuses the parent agent's model, so no separate model configuration is needed. The result: the agent always has a focused, relevant subset of tools for each request, even when hundreds of tools are registered on the gateway.

In [ ]:
from strands import Agent
from bedrock_agentcore.gateway.integrations.strands.plugins import AgentCoreToolSearchPlugin

# Create the agent with the AgentCore Tool Search Plugin
# The plugin uses the gateway's built-in semantic search to find relevant tools
# By default, intent classification reuses the agent's own model
agent = Agent(plugins=[
        AgentCoreToolSearchPlugin(
            mcp_client=mcp_client
        )
    ]
)

print("[OK] Strands Agent created with AgentCoreToolSearchPlugin")
print("   Semantic tool search powered by AgentCore Gateway")
print("   Intent classification reuses the agent model")
print("   Tools loaded dynamically before each model invocation")

### Customization Options

The `AgentCoreToolSearchPlugin` supports several customization options:

- **Custom model for intent classification** - Use a faster/cheaper model (e.g., Claude Haiku) via `StrandsIntentProvider(model=...)`
- **Custom system prompt** - Control how intent is derived from conversation history
- **Custom intent provider** - Subclass `IntentProvider` to implement your own intent derivation logic

For this tutorial, we use the default behavior (agent's own model for intent classification). See the [plugin source](https://github.com/aws/bedrock-agentcore-sdk-python/blob/main/src/bedrock_agentcore/gateway/integrations/strands/plugins/agentcore_tool_search) for full documentation and examples.

## 7. Agent Invocation Examples

Now that our agent is assembled, let's invoke it across multiple travel domains to see semantic tool search in action. After each invocation, we log the tools that the plugin loaded for that request - demonstrating how the gateway's `x_amz_bedrock_agentcore_search` tool finds only the relevant tools just before the model call, rather than loading all 34 tools every time.

This is the key benefit: each request gets a focused, relevant subset of tools based on the derived user intent.

### Flight Search

Let's start by asking the agent to find flights. The `ToolSearchPlugin` will classify this as a flight-related intent and load only the flight domain tools (search_flights, get_flight_details, check_availability, etc.) from the gateway.

In [ ]:
# Invoke the agent
response = agent("Find flights from San Francisco to New York next Friday")

# Log tools that were loaded by the plugin for this invocation
print("Tools selected by semantic search for this request:")
tools_config = agent.tool_registry.get_all_tools_config()
for tool_name in sorted(tools_config.keys()):
    print(f"   - {tool_name}")
print()
print("Agent Response:")
print(response)


### Hotel Search

Now let's switch to a completely different domain - hotels. Notice how the `ToolSearchPlugin` reclassifies the intent and selects hotel-related tools (search_hotels, get_hotel_details, check_room_availability, get_hotel_amenities) instead of flight tools.

In [ ]:
# Invoke the agent
response = agent("Search for hotels in Manhattan with a pool")

# Log tools that were loaded by the plugin for this invocation
print("Tools selected by semantic search for this request:")
tools_config = agent.tool_registry.get_all_tools_config()
for tool_name in sorted(tools_config.keys()):
    print(f"   - {tool_name}")
print()
print("Agent Response:")
print(response)


### Car Rental

Next, let's search for car rentals. The plugin will identify this as a car rental intent and load the relevant tools (search_car_rentals, get_rental_details, check_car_availability) from the gateway.

In [ ]:
# Invoke the agent
response = agent("Find available car rentals at JFK airport for next week")

# Log tools that were loaded by the plugin for this invocation
print("Tools selected by semantic search for this request:")
tools_config = agent.tool_registry.get_all_tools_config()
for tool_name in sorted(tools_config.keys()):
    print(f"   - {tool_name}")
print()
print("Agent Response:")
print(response)


### Restaurant Search

Let's try the restaurant domain. The plugin will select restaurant tools (search_restaurants, get_restaurant_details, get_menu, check_reservations, get_restaurant_reviews) for this request.

In [ ]:
# Invoke the agent
response = agent("Find Italian restaurants near Times Square with good reviews")

# Log tools that were loaded by the plugin for this invocation
print("Tools selected by semantic search for this request:")
tools_config = agent.tool_registry.get_all_tools_config()
for tool_name in sorted(tools_config.keys()):
    print(f"   - {tool_name}")
print()
print("Agent Response:")
print(response)


### Currency Conversion

Now let's try the currency domain. The plugin will identify this as a currency-related intent and load the currency tools (convert_currency, get_exchange_rates, get_supported_currencies).

In [ ]:
# Invoke the agent
response = agent("Convert 500 USD to EUR and show me the current exchange rate")

# Log tools that were loaded by the plugin for this invocation
print("Tools selected by semantic search for this request:")
tools_config = agent.tool_registry.get_all_tools_config()
for tool_name in sorted(tools_config.keys()):
    print(f"   - {tool_name}")
print()
print("Agent Response:")
print(response)


### Loyalty Program

Finally, let's check the loyalty program domain. The plugin will select loyalty tools (get_loyalty_balance, redeem_points, get_loyalty_program_info) for this request.

In [ ]:
# Invoke the agent
response = agent("Check my loyalty points balance and what rewards I can redeem")

# Log tools that were loaded by the plugin for this invocation
print("Tools selected by semantic search for this request:")
tools_config = agent.tool_registry.get_all_tools_config()
for tool_name in sorted(tools_config.keys()):
    print(f"   - {tool_name}")
print()
print("Agent Response:")
print(response)


## 8. Cleanup

We need to delete all resources created during this tutorial to avoid ongoing costs. Each cleanup step is independent - if one fails, others will still execute.

### Remove All Created Resources

We'll stop the MCP client connection and clean up all AWS resources created in this notebook: the Lambda function, its IAM execution role, and the AgentCore Gateway tool target registration.

In [ ]:
print("Cleaning up resources...")
print()

# 1. Stop MCP Client
try:
    mcp_client.__exit__(None, None, None)
    print("[OK] MCPClient connection stopped")
except Exception as e:
    print(f"[WARNING] Error stopping MCPClient: {e}")

# 2. Deregister from AgentCore Gateway
try:
    agentcore_control_client.delete_gateway_target(
        gatewayIdentifier=gateway_id,
        targetId=target_id
    )
    print("[OK] Deregistered tool target from gateway")
except Exception as e:
    print(f"[WARNING] Error deregistering from gateway: {e}")

# 3. Delete the AgentCore Gateway (wait for target deletion to propagate)
import time
time.sleep(10)
try:
    agentcore_control_client.delete_gateway(gatewayIdentifier=gateway_id)
    print(f"[OK] Deleted AgentCore Gateway: {GATEWAY_NAME}")
except Exception as e:
    print(f"[WARNING] Error deleting gateway: {e}")

# 4. Delete Lambda function
try:
    lambda_client.delete_function(FunctionName=LAMBDA_FUNCTION_NAME)
    print(f"[OK] Deleted Lambda function: {LAMBDA_FUNCTION_NAME}")
except Exception as e:
    print(f"[WARNING] Error deleting Lambda function: {e}")

# 5. Detach policy and delete Lambda IAM role
try:
    iam.detach_role_policy(
        RoleName=LAMBDA_ROLE_NAME,
        PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole"
    )
    iam.delete_role(RoleName=LAMBDA_ROLE_NAME)
    print(f"[OK] Deleted IAM role: {LAMBDA_ROLE_NAME}")
except Exception as e:
    print(f"[WARNING] Error deleting Lambda IAM role: {e}")

# 6. Detach policy and delete Gateway IAM role
try:
    iam.detach_role_policy(
        RoleName=GATEWAY_ROLE_NAME,
        PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaRole"
    )
    iam.delete_role(RoleName=GATEWAY_ROLE_NAME)
    print(f"[OK] Deleted IAM role: {GATEWAY_ROLE_NAME}")
except Exception as e:
    print(f"[WARNING] Error deleting Gateway IAM role: {e}")

print()
print("Cleanup complete! All resources have been removed.")


### Conclusion

In this notebook you built a Strands agent that uses the `AgentCoreToolSearchPlugin` to leverage AgentCore Gateway's semantic tool search. The plugin dynamically discovers and loads only the relevant tools before each model invocation - enabling efficient tool selection from 34 registered options without overwhelming the agent.
